In [1]:
from fundamental.commands import CommandRegistry, CommandSpec

In [2]:
class FakeApp:
    def __init__(self):
        self.messages = []
        self.opened = []
        self.closed = []

    def log(self, message):
        self.messages.append(message)

    def open_window(self, tag):
        self.opened.append(tag)

    def close_window(self, tag):
        self.closed.append(tag)

    def execute_command(self, command_text):
        return f"executed:{command_text}"


def test_register_and_execute_command():
    registry = CommandRegistry()
    app = FakeApp()

    def handler(ctx):
        return f"hello {ctx.args}"

    registry.register(
        CommandSpec(
            name="greet",
            description="Greet someone",
            handler=handler,
            aliases=("hi",),
        )
    )

    assert registry.execute("greet Alice", app) == "hello Alice"
    assert registry.execute("hi Bob", app) == "hello Bob"


def test_unknown_command():
    registry = CommandRegistry()
    app = FakeApp()

    assert registry.execute("missing", app) == "Unknown command: missing"


def test_empty_command():
    registry = CommandRegistry()
    app = FakeApp()

    assert registry.execute("   ", app) == "No command entered."


def test_context_helpers_delegate_to_app():
    registry = CommandRegistry()
    app = FakeApp()

    def handler(ctx):
        ctx.log("test log")
        ctx.open_window("tools.demo")
        ctx.close_window("tools.demo")
        return ctx.execute("nested.command")

    registry.register(
        CommandSpec(
            name="demo",
            description="Demo context helpers",
            handler=handler,
        )
    )

    assert registry.execute("demo", app) == "executed:nested.command"
    assert app.messages == ["test log"]
    assert app.opened == ["tools.demo"]
    assert app.closed == ["tools.demo"]


In [4]:
test_unknown_command()